# Supervised Learning Lab – Regression & Classification Pipeline

| | Regression | Classification |
|---|---|---|
| Data | Tips (`tip`) | Titanic (`survived`) |
| Models | Multiple Linear, Lasso (L1), Ridge (L2), Elastic Net | Logistic, Bagging, Random Forest, AdaBoost, Gradient Boosting, XGBoost |
| Evaluation | MSE, RMSE, MAE, MAPE | Confusion matrix, Precision, Recall, F1, ROC & AUC |

**Pipeline:** raw data → clean data → train/test split → `fit` → `predict` → evaluate → Excel export.

`SR` (serial number) is used **only for traceability** – it is never given to the model.

## 0. Install libraries (only if missing)

In [ ]:
import sys
import subprocess
import importlib.util


def ensure(module_name, pip_name):
    """If the module exists do nothing, otherwise pip install it."""
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])


ensure("pandas", "pandas")
ensure("matplotlib", "matplotlib")
ensure("sklearn", "scikit-learn")
ensure("openpyxl", "openpyxl")
ensure("xgboost", "xgboost")

## 1. Imports

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet, LogisticRegression
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                              AdaBoostClassifier, GradientBoostingClassifier)
from sklearn.metrics import (mean_squared_error, mean_absolute_error, mean_absolute_percentage_error,
                             confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score,
                             f1_score, accuracy_score, roc_auc_score, RocCurveDisplay)
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

## 2. Choose technique and algorithm (asked interactively, like the Streamlit sidebar)
Run the cell and type the number when prompted.
- Regularized models (Lasso, Ridge, Elastic Net) also ask for `alpha` and whether to **standardize** the features
- Elastic Net additionally asks for `l1_ratio`
- Press Enter to accept the default shown in brackets

In [ ]:
problem_menu = {"1": "Regression", "2": "Classification"}
reg_menu = {"1": "Multiple Linear Regression", "2": "Lasso Regression",
            "3": "Ridge Regression", "4": "Elastic Net Regression"}
clf_menu = {"1": "Logistic Regression", "2": "Bagging", "3": "Random Forest",
            "4": "AdaBoost", "5": "Gradient Boosting", "6": "XGBoost"}

PROBLEM = problem_menu[input("Supervised learning technique  [1] Regression  [2] Classification : ").strip()]

if PROBLEM == "Regression":
    DATA_NAME, TARGET = "tips", "tip"
    MODEL_NAME = reg_menu[input("Algorithm  [1] Multiple Linear  [2] Lasso  [3] Ridge  [4] Elastic Net : ").strip()]
else:
    DATA_NAME, TARGET = "titanic", "survived"
    MODEL_NAME = clf_menu[input("Algorithm  [1] Logistic  [2] Bagging  [3] Random Forest  "
                                "[4] AdaBoost  [5] Gradient Boosting  [6] XGBoost : ").strip()]

REGULARIZED = ["Lasso Regression", "Ridge Regression", "Elastic Net Regression"]

ALPHA, L1_RATIO, SCALE = 0.1, 0.5, False
if MODEL_NAME in REGULARIZED:
    ALPHA = float(input("alpha (regularization strength) [0.1] : ").strip() or 0.1)
    SCALE = (input("Standardize features? (y/n) [y] : ").strip().lower() or "y") == "y"
if MODEL_NAME == "Elastic Net Regression":
    L1_RATIO = float(input("l1_ratio [0.5] : ").strip() or 0.5)

TEST_SIZE = 0.30
RANDOM_STATE = 42

BASE_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/"
FOLDER = os.getcwd()          # files are saved next to this notebook

print("Selected:", PROBLEM, "-", MODEL_NAME)

## 3. Raw data
Load the data (downloaded once and cached as a CSV) and add `SR` – a serial number kept **only for traceability**.

In [ ]:
path = os.path.join(FOLDER, DATA_NAME + "_raw.csv")
if not os.path.exists(path):
    pd.read_csv(BASE_URL + DATA_NAME + ".csv").to_csv(path, index=False)

raw = pd.read_csv(path)
raw.insert(0, "SR", raw.index + 1)

print("Shape:", raw.shape)
print(raw.isnull().sum())
raw.head()

## 4. Clean data
- Missing **continuous** values → column **mean**
- Missing **categorical** values → column **mode**
- Titanic only: drop columns that repeat other columns (`class`, `who`, `adult_male`, `embark_town`, `alive`, `alone`) and `deck` (mostly empty)

In [ ]:
clean = raw.copy()
if DATA_NAME == "titanic":
    clean = clean.drop(columns=["class", "who", "adult_male", "deck", "embark_town", "alive", "alone"])

clean = clean.fillna(clean.mean(numeric_only=True))   # continuous -> mean
clean = clean.fillna(clean.mode().iloc[0])            # categorical -> mode

print("Missing values after cleaning:", int(clean.isnull().sum().sum()))
clean.head()

## 5. Train and test split
- `SR` and the target are separated **before** modelling, so `SR` never reaches the model
- Text columns are converted to 0/1 dummy columns
- 70 % train / 30 % test, `shuffle=True`, fixed `random_state`

In [ ]:
sr = clean["SR"]
y = clean[TARGET]
X = pd.get_dummies(clean.drop(columns=["SR", TARGET]), drop_first=True, dtype=int)

X_train, X_test, y_train, y_test, sr_train, sr_test = train_test_split(
    X, y, sr, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True)

print("Train:", X_train.shape, "  Test:", X_test.shape)

## 5b. Feature scaling (regularized models only)
Lasso, Ridge and Elastic Net penalize coefficient size, so features on different scales (e.g. `total_bill` vs `size`) are treated unfairly.
Standardizing (mean 0, std 1) fixes this. The scaler is **learnt on the training data only** and then applied to the test data.
The prediction tables and Excel keep the **original** values; only the model input is scaled.

In [ ]:
USE_SCALING = SCALE and MODEL_NAME in REGULARIZED

if USE_SCALING:
    scaler = StandardScaler()
    X_train_m = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
    X_test_m = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)
    print("Features standardized before fitting")
else:
    X_train_m = X_train
    X_test_m = X_test
    print("No scaling applied")

## 6. Model `fit` and `predict`

In [ ]:
all_models = {
    "Multiple Linear Regression": LinearRegression(),
    "Lasso Regression": Lasso(alpha=ALPHA),
    "Ridge Regression": Ridge(alpha=ALPHA),
    "Elastic Net Regression": ElasticNet(alpha=ALPHA, l1_ratio=L1_RATIO),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Bagging": BaggingClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostClassifier(random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss"),
}
model = all_models[MODEL_NAME]

model.fit(X_train_m, y_train)

train_pred = model.predict(X_train_m)
test_pred = model.predict(X_test_m)

## 7. Prediction tables (with `SR`)
`SR` comes first for traceability. Classification also gets class probabilities.
If scaling was used, the **standardized** train/test tables (what the model really saw) and the scaler's mean/std are built here too.

In [ ]:
train_out = X_train.copy()
test_out = X_test.copy()
train_out.insert(0, "SR", sr_train)
test_out.insert(0, "SR", sr_test)
train_out["Actual"] = y_train
test_out["Actual"] = y_test
train_out["Predicted"] = train_pred
test_out["Predicted"] = test_pred

if PROBLEM == "Classification":
    train_prob = model.predict_proba(X_train_m)
    test_prob = model.predict_proba(X_test_m)
    train_out["Prob_Class_0"] = train_prob[:, 0]
    train_out["Prob_Class_1"] = train_prob[:, 1]
    test_out["Prob_Class_0"] = test_prob[:, 0]
    test_out["Prob_Class_1"] = test_prob[:, 1]

# the exact standardized data the model was trained / tested on (only when scaling is used)
if USE_SCALING:
    train_scaled_out = X_train_m.copy()
    test_scaled_out = X_test_m.copy()
    train_scaled_out.insert(0, "SR", sr_train)
    test_scaled_out.insert(0, "SR", sr_test)
    train_scaled_out["Actual"] = y_train
    test_scaled_out["Actual"] = y_test
    train_scaled_out["Predicted"] = train_pred
    test_scaled_out["Predicted"] = test_pred
    scaler_df = pd.DataFrame({"Feature": X.columns, "Mean (train)": scaler.mean_, "Std Dev (train)": scaler.scale_})

test_out.head()

## 8. Coefficients / feature importance
- Linear models → coefficients + intercept
- Tree/boosting models → `feature_importances_`
- Bagging (has neither) → permutation importance on the test set

In [ ]:
if hasattr(model, "coef_"):
    coef_name = "Coefficient (standardized features)" if USE_SCALING else "Coefficient"
    coefs = pd.DataFrame({"Feature": X.columns, coef_name: model.coef_.ravel()})
    coefs.loc[len(coefs)] = ["Intercept", float(model.intercept_.ravel()[0])]
elif hasattr(model, "feature_importances_"):
    coefs = pd.DataFrame({"Feature": X.columns, "Importance": model.feature_importances_})
else:
    perm = permutation_importance(model, X_test_m, y_test, random_state=RANDOM_STATE)
    coefs = pd.DataFrame({"Feature": X.columns, "Importance (permutation)": perm.importances_mean})

coefs

## 9. Model evaluation
- **Regression:** MSE, RMSE, MAE, MAPE
- **Classification:** Accuracy, Precision, Recall, F1, AUC and the confusion matrix
- Train vs Test side by side – a big gap means over-fitting

In [ ]:
cm_df = None

if PROBLEM == "Regression":
    train_metrics = {
        "MSE": mean_squared_error(y_train, train_pred),
        "RMSE": mean_squared_error(y_train, train_pred) ** 0.5,
        "MAE": mean_absolute_error(y_train, train_pred),
        "MAPE (%)": mean_absolute_percentage_error(y_train, train_pred) * 100,
    }
    test_metrics = {
        "MSE": mean_squared_error(y_test, test_pred),
        "RMSE": mean_squared_error(y_test, test_pred) ** 0.5,
        "MAE": mean_absolute_error(y_test, test_pred),
        "MAPE (%)": mean_absolute_percentage_error(y_test, test_pred) * 100,
    }
else:
    train_metrics = {
        "Accuracy": accuracy_score(y_train, train_pred),
        "Precision": precision_score(y_train, train_pred),
        "Recall": recall_score(y_train, train_pred),
        "F1 score": f1_score(y_train, train_pred),
        "AUC": roc_auc_score(y_train, train_prob[:, 1]),
    }
    test_metrics = {
        "Accuracy": accuracy_score(y_test, test_pred),
        "Precision": precision_score(y_test, test_pred),
        "Recall": recall_score(y_test, test_pred),
        "F1 score": f1_score(y_test, test_pred),
        "AUC": roc_auc_score(y_test, test_prob[:, 1]),
    }
    cm = confusion_matrix(y_test, test_pred)
    cm_df = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"])

metrics = pd.DataFrame({"Train": train_metrics, "Test": test_metrics})
metrics

## 10. Plots
- Regression: actual vs predicted (points on the red line = perfect prediction)
- Classification: confusion matrix and ROC curve on the test set

In [ ]:
if PROBLEM == "Regression":
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(y_test, test_pred)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.set_title("Test: actual vs predicted")
    plt.show()
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
    ConfusionMatrixDisplay(cm).plot(ax=ax1, colorbar=False)
    ax1.set_title("Confusion matrix (test)")
    RocCurveDisplay.from_estimator(model, X_test_m, y_test, ax=ax2)
    ax2.plot([0, 1], [0, 1], "k--")
    ax2.set_title("ROC curve (test)")
    plt.show()

## 11. Export to Excel
File name follows the model, e.g. `Lasso_Regression.xlsx`, and is saved in the `Output` subfolder (created automatically) so raw data and results stay separate.

| Sheet | Content |
|---|---|
| Raw_Data | original data as-is, with `SR` |
| Training | `SR`, features, actual, predicted (+ probabilities) |
| Testing | same as above for the test rows |
| Training_Standardized, Testing_Standardized, Scaler_Mean_Std | *only when scaling is used:* the standardized data the model actually saw, and the mean / std dev learnt from the training data |
| Coefficients | coefficients + intercept, or feature importance |
| Evaluation | metrics (+ `Confusion_Matrix` sheet for classification) |

In [ ]:
file_name = MODEL_NAME.replace(" ", "_") + ".xlsx"
output_folder = os.path.join(FOLDER, "Output")
os.makedirs(output_folder, exist_ok=True)
file_path = os.path.join(output_folder, file_name)

with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
    raw.to_excel(writer, sheet_name="Raw_Data", index=False)
    train_out.to_excel(writer, sheet_name="Training", index=False)
    test_out.to_excel(writer, sheet_name="Testing", index=False)
    if USE_SCALING:
        train_scaled_out.to_excel(writer, sheet_name="Training_Standardized", index=False)
        test_scaled_out.to_excel(writer, sheet_name="Testing_Standardized", index=False)
        scaler_df.to_excel(writer, sheet_name="Scaler_Mean_Std", index=False)
    coefs.to_excel(writer, sheet_name="Coefficients", index=False)
    metrics.to_excel(writer, sheet_name="Evaluation")
    if cm_df is not None:
        cm_df.to_excel(writer, sheet_name="Confusion_Matrix")

print("Saved:", file_path)